In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import yaml
from datetime import datetime
from pathlib import Path

from src.data.load_daily import load_daily

# ETF 数据质量检查 (Data QC)

**目标**: 对下载的 ETF 数据进行质量检查，识别潜在问题

检查内容：
1. 每只 ETF 可用交易日数量、缺失比例
2. 硬错误计数：close<=0、high<low、volume<0
3. 极端日收益计数：abs(ret_1) > 0.1

## 1. 加载数据

In [2]:
# 读取配置
with open('../configs/universe.yaml', 'r', encoding='utf-8') as f:
    universe_config = yaml.safe_load(f)

with open('../configs/experiment.yaml', 'r', encoding='utf-8') as f:
    exp_config = yaml.safe_load(f)

# 构建完整资产池
universe = []
for category in universe_config.values():
    universe.extend(category)

print(f"资产池规模: {len(universe)} 只 ETF")
print(f"回测区间: {exp_config['start_date']} 至 {exp_config['end_date']}")

资产池规模: 29 只 ETF
回测区间: 2020-01-01 至 2025-12-31


In [3]:
# 加载数据
print("开始下载数据...")
df = load_daily(
    universe=universe,
    start_date=exp_config['start_date'],
    end_date=exp_config['end_date'],
    adjust='qfq',
    retry=3,
    sleep=0.5
)

print(f"数据形状: {df.shape}")
print(f"列名: {df.columns.tolist()}")
df.head()

开始下载数据...
✓ 510300.SH: 1455 records
✓ 510500.SH: 1455 records
✓ 159915.SZ: 1454 records
✓ 512880.SH: 1455 records
✓ 512690.SH: 1454 records
✓ 515000.SH: 1455 records
✓ 159949.SZ: 1455 records
✓ 512480.SH: 1454 records
✓ 515790.SH: 1222 records
✓ 516160.SH: 1189 records
✓ 512760.SH: 1454 records
✓ 512010.SH: 1454 records
✓ 159928.SZ: 1455 records
✓ 515880.SH: 1455 records
✓ 159745.SZ: 1103 records
✓ 512400.SH: 1455 records
✓ 512200.SH: 1455 records
✓ 515220.SH: 1419 records
✓ 518880.SH: 1455 records
✓ 159980.SZ: 1455 records
✓ 159985.SZ: 1455 records
✓ 159981.SZ: 1444 records
✓ 511010.SH: 1455 records
✓ 511260.SH: 1455 records
✓ 511220.SH: 1455 records
✓ 511880.SH: 1455 records
✓ 511990.SH: 1455 records
✓ 511660.SH: 1455 records

Total symbols: 29
Successfully loaded: 29
Failed: 0
Date range: 2020-01-02 to 2025-12-31
Total records: 39837

数据形状: (39837, 8)
列名: ['date', 'symbol', 'open', 'high', 'low', 'close', 'volume', 'amount']


,date,symbol,open,high,low,close,volume,amount
0,2020-01-02,159915.SZ,1.743,1.771,1.731,1.765,3380825.0,593821312.0
1,2020-01-02,159928.SZ,0.761,0.762,0.748,0.756,195003.0,58905887.0
2,2020-01-02,159949.SZ,0.666,0.677,0.659,0.673,5200278.0,348810496.0
3,2020-01-02,159980.SZ,1.013,1.015,1.008,1.009,75622.0,7641714.0
4,2020-01-02,159985.SZ,0.989,0.992,0.986,0.988,118014.0,11672903.0


## 2. 检查1：交易日数量与缺失比例

In [4]:
# 获取完整交易日列表（使用数据最多的ETF作为基准）
all_dates = df['date'].unique()
total_trading_days = len(all_dates)
print(f"总交易日数: {total_trading_days}")

# 统计每只ETF的交易日数量
coverage_stats = df.groupby('symbol').agg({
    'date': 'count',
    'close': 'count'
}).rename(columns={'date': 'trading_days', 'close': 'data_points'})

coverage_stats['missing_rate'] = 1 - coverage_stats['trading_days'] / total_trading_days
coverage_stats = coverage_stats.sort_values('trading_days', ascending=False)

print(f"\n交易日数量统计:")
print(coverage_stats.describe())
print(f"\n缺失率 > 5% 的 ETF:")
print(coverage_stats[coverage_stats['missing_rate'] > 0.05])

总交易日数: 1455

交易日数量统计:
       trading_days  data_points  missing_rate
count     28.000000    28.000000     28.000000
mean    1422.750000  1422.750000      0.022165
std       90.523017    90.523017      0.062215
min     1103.000000  1103.000000      0.000000
25%     1454.000000  1454.000000      0.000000
50%     1455.000000  1455.000000      0.000000
75%     1455.000000  1455.000000      0.000687
max     1455.000000  1455.000000      0.241924

缺失率 > 5% 的 ETF:
           trading_days  data_points  missing_rate
symbol                                            
515790.SH          1222         1222      0.160137
516160.SH          1189         1189      0.182818
159745.SZ          1103         1103      0.241924


## 3. 检查2：硬错误检测

In [5]:
# 硬错误检测
errors = pd.DataFrame()

# 错误1: close <= 0
errors['close_non_positive'] = (df['close'] <= 0).astype(int)

# 错误2: high < low
errors['high_lt_low'] = (df['high'] < df['low']).astype(int)

# 错误3: volume < 0
errors['volume_negative'] = (df['volume'] < 0).astype(int)

# 添加 symbol 列用于分组统计
errors['symbol'] = df['symbol']

# 汇总统计
error_summary = errors.groupby('symbol').sum()
total_errors = errors[['close_non_positive', 'high_lt_low', 'volume_negative']].sum()

print("=== 硬错误统计 ===")
print(f"Close <= 0 的记录数: {total_errors['close_non_positive']}")
print(f"High < Low 的记录数: {total_errors['high_lt_low']}")
print(f"Volume < 0 的记录数: {total_errors['volume_negative']}")
print(f"\n各ETF错误分布:")
print(error_summary[error_summary.sum(axis=1) > 0])

=== 硬错误统计 ===
Close <= 0 的记录数: 0
High < Low 的记录数: 0
Volume < 0 的记录数: 0

各ETF错误分布:
Empty DataFrame
Columns: [close_non_positive, high_lt_low, volume_negative]
Index: []


## 4. 检查3：极端日收益

In [6]:
# 计算日收益率
df_sorted = df.sort_values(['symbol', 'date'])
df_sorted['ret_1'] = df_sorted.groupby('symbol')['close'].pct_change()

# 识别极端收益
extreme_threshold = 0.1
df_sorted['extreme_ret'] = (df_sorted['ret_1'].abs() > extreme_threshold).astype(int)

# 统计极端收益
extreme_count = df_sorted.groupby('symbol')['extreme_ret'].sum()
extreme_count = extreme_count[extreme_count > 0].sort_values(ascending=False)

print(f"=== 极端日收益统计 (|ret_1| > {extreme_threshold}) ===")
print(f"总极端收益记录数: {df_sorted['extreme_ret'].sum()}")
print(f"\n各ETF极端收益次数:")
print(extreme_count)

# 展示具体的极端收益案例
print(f"\n极端收益案例（前10条）:")
extreme_cases = df_sorted[df_sorted['extreme_ret'] == 1][['date', 'symbol', 'close', 'ret_1']].head(10)
print(extreme_cases)

=== 极端日收益统计 (|ret_1| > 0.1) ===
总极端收益记录数: 61

各ETF极端收益次数:
symbol
512690.SH    17
512760.SH     6
159915.SZ     5
159949.SZ     5
512880.SH     5
512200.SH     3
512400.SH     3
515790.SH     3
515880.SH     3
159745.SZ     2
159928.SZ     2
512010.SH     2
515220.SH     2
516160.SH     2
512480.SH     1
Name: extreme_ret, dtype: int32

极端收益案例（前10条）:
            date     symbol  close     ret_1
31241 2024-09-26  159745.SZ  0.545  0.101010
31297 2024-09-30  159745.SZ  0.622  0.100885
31270 2024-09-27  159915.SZ  1.860  0.103203
31298 2024-09-30  159915.SZ  2.232  0.200000
31326 2024-10-08  159915.SZ  2.678  0.199821
31354 2024-10-09  159915.SZ  2.242 -0.162808
34714 2025-04-07  159915.SZ  1.775 -0.124322
374   2020-02-03  159928.SZ  0.654 -0.100413
31299 2024-09-30  159928.SZ  0.934  0.100118
31272 2024-09-27  159949.SZ  0.841  0.115385
